# Week 3: Exploratory Data Analysis & Feature Engineering
## Notebook 11: Weight of Evidence (WoE) & Information Value (IV)

### 1. Conceptual Intuition & Theoretical Background
In banking and credit risk modeling (e.g. Basel regulatory scorecards), features must be monotonic, interpretable, and robust to outliers.

**Weight of Evidence (WoE):**
Measures the relative log-odds of a bin separating "Good" customers (non-defaulters) from "Bad" customers (defaulters):
$$\text{WoE}_i = \ln\left( \frac{\%\text{ Goods}_i}{\%\text{ Bads}_i} \right) = \ln\left( \frac{G_i / G_{\text{total}}}{B_i / B_{\text{total}}} \right)$$

**Information Value (IV):**
Quantifies a feature's overall predictive power across all $K$ bins:
$$\text{IV} = \sum_{i=1}^{K} \left( \frac{G_i}{G_{\text{total}}} - \frac{B_i}{B_{\text{total}}} \right) \times \text{WoE}_i$$

**Regulatory Predictive Power Rules of Thumb:**
- $\text{IV} < 0.02$: Unpredictive; discard feature.
- $0.02 \le \text{IV} < 0.1$: Weak predictor.
- $0.1 \le \text{IV} < 0.3$: Medium predictor.
- $0.3 \le \text{IV} < 0.5$: Strong predictor.
- $\text{IV} \ge 0.5$: Suspiciously high; check for target leakage!

---

### 2. Scratch Python Implementation: WoE & IV Calculator

In [ ]:
import numpy as np
import pandas as pd

def calculate_woe_iv(df, feature, target, num_bins=5):
    """
    Computes Weight of Evidence (WoE) and Information Value (IV).
    target: 1 for Bad (Default), 0 for Good (Non-default)
    """
    # Create quantile bins
    df_temp = df[[feature, target]].copy()
    df_temp['bin'] = pd.qcut(df_temp[feature], q=num_bins, duplicates='drop')
    
    # Aggregation
    grouped = df_temp.groupby('bin', observed=False)[target].agg(
        total='count',
        bads='sum'
    )
    grouped['goods'] = grouped['total'] - grouped['bads']
    
    total_goods = grouped['goods'].sum()
    total_bads = grouped['bads'].sum()
    
    # Distributions with epsilon to avoid division by zero
    grouped['dist_goods'] = (grouped['goods'] / total_goods) + 1e-6
    grouped['dist_bads'] = (grouped['bads'] / total_bads) + 1e-6
    
    # Calculate WoE and IV per bin
    grouped['WoE'] = np.log(grouped['dist_goods'] / grouped['dist_bads'])
    grouped['IV_bin'] = (grouped['dist_goods'] - grouped['dist_bads']) * grouped['WoE']
    
    total_iv = grouped['IV_bin'].sum()
    return grouped[['total', 'goods', 'bads', 'WoE', 'IV_bin']], total_iv

# Generate Synthetic Loan Credit Dataset
np.random.seed(42)
n_applicants = 1000
# Feature 1: Debt-to-Income Ratio (Strong Predictor)
dti = np.random.uniform(0.1, 0.8, n_applicants)
prob_default_dti = 1.0 / (1.0 + np.exp(-(8.0 * dti - 3.5)))
target_dti = np.random.binomial(1, prob_default_dti)

# Feature 2: Random Uncorrelated Noise
noise_feat = np.random.normal(50, 10, n_applicants)

df_credit = pd.DataFrame({'DTI': dti, 'Noise': noise_feat, 'Default': target_dti})

table_dti, iv_dti = calculate_woe_iv(df_credit, 'DTI', 'Default')
table_noise, iv_noise = calculate_woe_iv(df_credit, 'Noise', 'Default')

print("--- Credit Risk Scorecard: DTI Feature Analysis ---")
print(table_dti[['goods', 'bads', 'WoE', 'IV_bin']])
print(f"\nTotal DTI Information Value (IV):   {iv_dti:.4f} (Strong Predictor)")
print(f"Total Noise Information Value (IV): {iv_noise:.4f} (Unpredictive < 0.02)")

### 3. Step-by-Step Code Walkthrough
1. **Quantile Binning:** Slices continuous features into balanced bins.
2. **WoE Computation:** Positive WoE indicates a higher proportion of goods than bads.
3. **Information Value (IV):** Integrates symmetric divergence; accurately identifies DTI as a strong predictor while rejecting noise.

---

### 4. Industry Applications & Trade-offs
- **Credit Bureau Scorecards (FICO):** Regulated risk models replace continuous features with their discrete WoE values to ensure strictly monotonic linear relationships with default log-odds.
- **Trade-off:** Quantile binning discards fine granular variance within individual bins.

---

### 5. Top Interview Questions & Answers
**Q1: Why does WoE transformation linearize the relationship with logistic regression?**
> **Answer:** Because $\text{WoE} = \ln(\%\text{Goods} / \%\text{Bads})$ directly corresponds to the log-odds scale modeled by logistic regression, turning non-linear relationships into linear ones.

**Q2: What does an Information Value (IV) greater than 0.5 indicate?**
> **Answer:** An IV above 0.5 suggests potential target leakage (e.g. including an account status feature determined after loan default).